# Window & Wall Segmentation (Colab) — SAM-based, Ready-to-Run

This notebook uses **Meta's Segment Anything (SAM)** to generate high-quality masks, then applies **robust heuristics** to classify masks into **windows** vs **walls** (plus *other*).

✅ Works on **single images** or **folders**  
✅ Produces **publication-friendly overlays** + **binary masks**  
✅ No API keys; runs locally in Colab

---

## How to use
1. Run **Setup** (installs + downloads checkpoint).
2. Set `INPUT_DIR` (folder) or `INPUT_IMAGE` (single file).
3. Run **Single Image Test** first.
4. Run **Batch Segmentation**.
5. Check outputs under `OUTPUT_DIR`.


In [ ]:
# =========================
# 0) SETUP (RUN THIS CELL)
# =========================

!pip -q install git+https://github.com/facebookresearch/segment-anything.git
!pip -q install opencv-python matplotlib pillow tqdm scikit-image numpy

import torch
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

import pathlib
CHECKPOINT_DIR = pathlib.Path("checkpoints")
CHECKPOINT_DIR.mkdir(exist_ok=True)

SAM_MODEL_TYPE = "vit_b"  # options: vit_b, vit_l, vit_h
SAM_CKPT = CHECKPOINT_DIR / "sam_vit_b_01ec64.pth"

if not SAM_CKPT.exists():
    !wget -q -O checkpoints/sam_vit_b_01ec64.pth \
      https://dl.fbaipublicfiles.com/segment_anything/sam_vit_b_01ec64.pth

print("Checkpoint ready:", SAM_CKPT.exists(), str(SAM_CKPT))


  Preparing metadata (setup.py) ... done
PyTorch: 2.9.0+cpu
CUDA available: False
Checkpoint ready: True checkpoints/sam_vit_b_01ec64.pth


---
## 1) Imports + Model Load


In [ ]:
import numpy as np
import cv2
import matplotlib.pyplot as plt
from pathlib import Path
from tqdm import tqdm

import torch
from segment_anything import sam_model_registry, SamAutomaticMaskGenerator

np.random.seed(42)

device = "cuda" if torch.cuda.is_available() else "cpu"

sam = sam_model_registry[SAM_MODEL_TYPE](checkpoint=str(SAM_CKPT))
sam.to(device=device)

mask_generator = SamAutomaticMaskGenerator(
    model=sam,
    points_per_side=32,
    pred_iou_thresh=0.90,
    stability_score_thresh=0.92,
    crop_n_layers=1,
    crop_n_points_downscale_factor=2,
    min_mask_region_area=500
)

print("SAM loaded on:", device)


---
## 2) Helper Functions (Classification + Visualization)

We classify masks using façade-friendly cues:
- **Rectangularity**
- **Aspect ratio**
- **Area ratio**
- **Edge density**

These are **heuristics** (no training required). For a publication, label ~50 masks and train a small classifier on these same features.


In [ ]:
def mask_to_bbox(mask: np.ndarray):
    ys, xs = np.where(mask)
    if xs.size == 0:
        return None
    x0, x1 = int(xs.min()), int(xs.max())
    y0, y1 = int(ys.min()), int(ys.max())
    return x0, y0, x1, y1

def rectangularity(mask: np.ndarray):
    bbox = mask_to_bbox(mask)
    if bbox is None:
        return 0.0
    x0, y0, x1, y1 = bbox
    bbox_area = max(1, (x1 - x0 + 1) * (y1 - y0 + 1))
    return float(mask.sum()) / float(bbox_area)

def aspect_ratio(mask: np.ndarray):
    bbox = mask_to_bbox(mask)
    if bbox is None:
        return 0.0
    x0, y0, x1, y1 = bbox
    w = (x1 - x0 + 1)
    h = (y1 - y0 + 1)
    return float(w) / float(h) if h > 0 else 0.0

def edge_density(gray: np.ndarray, mask: np.ndarray):
    bbox = mask_to_bbox(mask)
    if bbox is None:
        return 0.0
    x0, y0, x1, y1 = bbox
    crop = gray[y0:y1+1, x0:x1+1]
    m = mask[y0:y1+1, x0:x1+1].astype(np.uint8)
    if crop.size == 0 or not (m > 0).any():
        return 0.0
    edges = cv2.Canny(crop, 50, 150)
    return float((edges > 0)[m > 0].mean())

def classify_mask(mask: np.ndarray, img_shape, gray=None):
    H, W = img_shape[:2]
    area_ratio = float(mask.sum()) / float(H * W)

    rect = rectangularity(mask)
    ar = aspect_ratio(mask)
    ed = edge_density(gray, mask) if gray is not None else 0.0

    # window
    if (rect >= 0.70) and (0.35 <= ar <= 3.0) and (0.001 <= area_ratio <= 0.15) and (ed >= 0.05):
        return "window"

    # wall
    if ((area_ratio >= 0.20) and (ed <= 0.04)) or ((area_ratio >= 0.08) and (ed <= 0.04) and (rect < 0.70)):
        return "wall"

    return "other"

def overlay_masks(image_bgr, masks_by_class, alpha=0.55):
    img = image_bgr.copy()
    colors = {"window": (255, 255, 0), "wall": (0, 255, 255), "other": (255, 0, 255)}  # BGR
    overlay = img.copy()
    for cls, masks in masks_by_class.items():
        for m in masks:
            overlay[m] = colors[cls]
    blended = cv2.addWeighted(overlay, alpha, img, 1 - alpha, 0)
    return cv2.cvtColor(blended, cv2.COLOR_BGR2RGB)

def combine_masks(masks_list, shape):
    out = np.zeros(shape[:2], dtype=np.uint8)
    for m in masks_list:
        out[m] = 255
    return out


---
## 3) Configure Inputs / Outputs


In [ ]:
# =========================
# 3) CONFIGURE INPUTS / OUTPUTS
# =========================

INPUT_IMAGE = ""  # e.g. "/content/my_image.jpg"
INPUT_DIR   = ""  # e.g. "/content/drive/MyDrive/Research/facade_images"

OUTPUT_DIR = "/content/seg_outputs"
Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

EXTS = {".jpg", ".jpeg", ".png", ".tif", ".tiff"}

print("OUTPUT_DIR:", OUTPUT_DIR)


---
## 4) Single Image Test


In [ ]:
def process_image(path: str, save_outputs=True, show=True):
    path = str(path)
    bgr = cv2.imread(path)
    if bgr is None:
        raise FileNotFoundError(f"Could not read image: {path}")
    gray = cv2.cvtColor(bgr, cv2.COLOR_BGR2GRAY)
    rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)

    masks = sorted(mask_generator.generate(rgb), key=lambda x: x["area"], reverse=True)

    masks_by_class = {"window": [], "wall": [], "other": []}
    for m in masks:
        seg = m["segmentation"].astype(bool)
        cls = classify_mask(seg, bgr.shape, gray=gray)
        masks_by_class[cls].append(seg)

    overlay = overlay_masks(bgr, masks_by_class, alpha=0.55)
    win_mask = combine_masks(masks_by_class["window"], bgr.shape)
    wall_mask = combine_masks(masks_by_class["wall"], bgr.shape)

    if save_outputs:
        stem = Path(path).stem
        cv2.imwrite(str(Path(OUTPUT_DIR) / f"{stem}__overlay.png"), cv2.cvtColor(overlay, cv2.COLOR_RGB2BGR))
        cv2.imwrite(str(Path(OUTPUT_DIR) / f"{stem}__window_mask.png"), win_mask)
        cv2.imwrite(str(Path(OUTPUT_DIR) / f"{stem}__wall_mask.png"), wall_mask)

    if show:
        plt.figure(figsize=(15,6))
        plt.subplot(1,3,1); plt.title("Original"); plt.axis("off")
        plt.imshow(cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB))
        plt.subplot(1,3,2); plt.title("Overlay (Window/Wall/Other)"); plt.axis("off")
        plt.imshow(overlay)
        plt.subplot(1,3,3); plt.title("Window Mask"); plt.axis("off")
        plt.imshow(win_mask, cmap="gray")
        plt.tight_layout()
        plt.show()

    return masks_by_class

if INPUT_IMAGE.strip():
    _ = process_image(INPUT_IMAGE, save_outputs=True, show=True)
else:
    print("Set INPUT_IMAGE above to test a single image.")


---
## 5) Batch Processing


In [ ]:
def list_images(folder):
    folder = Path(folder)
    return sorted([p for p in folder.rglob("*") if p.suffix.lower() in EXTS])

def batch_process(input_dir: str):
    paths = list_images(input_dir)
    if not paths:
        raise FileNotFoundError(f"No images found in: {input_dir}")
    print(f"Found {len(paths)} images. Processing...")

    for p in tqdm(paths):
        try:
            process_image(str(p), save_outputs=True, show=False)
        except Exception as e:
            print(f"[WARN] Failed on {p.name}: {e}")

    print("Done. Outputs saved to:", OUTPUT_DIR)

if INPUT_DIR.strip():
    batch_process(INPUT_DIR)
else:
    print("Set INPUT_DIR above to run batch processing.")


---
## 6) Tuning Tips

- Too many tiny fragments → increase `min_mask_region_area` and/or raise `stability_score_thresh`
- Missing windows/panes → increase `points_per_side` (slower) and/or lower `pred_iou_thresh` slightly
- Windows misclassified → relax thresholds in `classify_mask`

For a paper: label ~50 masks and train a small classifier on these mask features; report F1/accuracy.
